# Transformer-WAF — Q1 Experiments (one notebook)

Runs every experiment in `research/README.md` in order. **Run the cells from top to bottom.**

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU***.
2. Have the CSIC 2010 dataset ready (Section 2 shows three ways to get it).
3. Optional for transfer tests: SR-BH 2020 and the 2025 ModSecurity capture (Section 3).

**Tips**
- Results are saved to Google Drive (Section 1), so a Colab disconnect does not lose work. Every step **skips work that is already done**, so after a disconnect just run all cells again.
- Each section has an on/off switch in the *Settings* cell.
- Nothing here invents numbers: every table in the paper should come from the files this notebook writes.

## 1. Setup — GPU check, Google Drive, code

In [ ]:
import os, subprocess, sys, json, glob, shutil

# ---- GPU check ----
try:
    import torch
    print("CUDA available:", torch.cuda.is_available(),
          "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
except ImportError:
    print("torch will be installed below")

# ---- Save everything on Google Drive (recommended) ----
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/waf_q1'
else:
    WORK = '/content/waf_q1'
for d in ['data/raw', 'data/processed', 'data/splits', 'models', 'results', 'configs']:
    os.makedirs(f'{WORK}/{d}', exist_ok=True)

# ---- Get the code ----
# The repository is private, so Colab needs one of these:
#   A) a GitHub token (github.com -> Settings -> Developer settings ->
#      Personal access tokens -> Fine-grained, read access to this repo), or
#   B) a zip of the project on your Drive at MyDrive/waf_q1/transformer-waf-paper.zip
REPO = '/content/transformer-waf-paper'
URL = 'github.com/PriscillajospinG/transformer-waf-paper.git'
ZIP = f'{WORK}/transformer-waf-paper.zip'
if not os.path.exists(REPO):
    if os.path.exists(ZIP):
        subprocess.run(['unzip', '-q', ZIP, '-d', '/content/_code'], check=True)
        inner = [d for d in glob.glob('/content/_code/*') if os.path.isdir(d)]
        shutil.move(inner[0] if len(inner) == 1 and os.path.exists(f'{inner[0]}/research') else '/content/_code', REPO)
        print("code taken from", ZIP)
    else:
        r = subprocess.run(['git', 'clone', '-q', 'https://' + URL, REPO])
        if r.returncode != 0:
            from getpass import getpass
            tok = getpass("Repo is private: paste a GitHub token (input is hidden): ").strip()
            subprocess.run(['git', 'clone', '-q', f'https://{tok}@{URL}', REPO], check=True)
            subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', 'https://' + URL])
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=False)
assert os.path.exists(f'{REPO}/research/train.py'), "code not found - see the comments above"
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO}/research/requirements.txt'], check=True)
R = f'{REPO}/research'
print("WORK =", WORK)

In [ ]:
# ---- Helpers: run a script, skip if its output already exists ----
def sh(args, out=None, force=False):
    '''Run `python <args>`; skip if `out` exists (so re-running resumes).'''
    if out and os.path.exists(out) and not force:
        print("skip (done):", os.path.relpath(out, WORK)); return
    cmd = [sys.executable] + [str(a) for a in args]
    print("\n$ python", os.path.basename(cmd[1]), " ".join(cmd[2:]))
    p = subprocess.run(cmd, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"command failed (exit {p.returncode}) — read the error above")

def P(*parts):            # path inside WORK
    return os.path.join(WORK, *parts)

def show(path, keys=('precision', 'recall', 'f1', 'fpr', 'roc_auc')):
    m = json.load(open(path))['metrics']
    print(os.path.relpath(path, WORK), {k: m.get(k) for k in keys})

## Settings — switch sections on/off

In [ ]:
SEEDS        = [42, 1, 7, 13, 21]   # 5 seeds for mean ± std. Start with [42] for a quick first run.
BASE_MODEL   = 'bert-base-uncased'
MODEL_INPUT  = 'decoded'            # matches the live WAF service
EPOCHS, BATCH = 3, 16               # use BATCH = 8 if you get "CUDA out of memory"

RUN_BASELINES   = True
RUN_THRESHOLD   = True
RUN_STATS       = True
RUN_CALIBRATION = True
RUN_LEARNED     = True
RUN_ADVERSARIAL = True
RUN_LATENCY     = True
RUN_ENCODERS    = True   # SecureBERT + DistilBERT (one seed each, ~40 min extra)
RUN_TRANSFER    = True   # needs SR-BH and/or the 2025 corpus (Section 3)
RUN_REVERSE     = True   # train on SR-BH, test on CSIC (needs SR-BH)

## 2. CSIC 2010 dataset

Use **one** option. You need either the two raw files `normalTrafficTraining.txt` + `anomalousTrafficTest.txt`, **or** one CSV (e.g. the Kaggle "CSIC 2010" CSV).

In [ ]:
# ---- Option A: upload from your computer ----
UPLOAD_CSIC = False          # set True and run, then pick the file(s)
if UPLOAD_CSIC:
    from google.colab import files
    for name, data in files.upload().items():
        open(P('data/raw', name), 'wb').write(data); print("saved", name)
print("data/raw:", os.listdir(P('data/raw')))

In [ ]:
# ---- Option B: Kaggle API (needs your kaggle.json from kaggle.com -> Settings -> API) ----
USE_KAGGLE = False
KAGGLE_DATASET = 'ispangler/csic-2010-web-application-attacks'   # check the name on kaggle.com
if USE_KAGGLE:
    from google.colab import files
    files.upload()                                   # choose kaggle.json
    os.makedirs('/root/.kaggle', exist_ok=True); shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
    os.chmod('/root/.kaggle/kaggle.json', 0o600)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'kaggle'], check=True)
    subprocess.run(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET, '-p', P('data/raw'), '--unzip'], check=True)
print("data/raw:", os.listdir(P('data/raw')))

In [ ]:
# ---- Option C: try a public GitHub mirror of the raw files (may be offline) ----
TRY_MIRROR = False
if TRY_MIRROR:
    import urllib.request
    base = 'https://raw.githubusercontent.com/lucasmsp/machine-learning/master/DecisionTree/HTTP-dataset-CSIC-2010/'
    for f in ['normalTrafficTraining.txt', 'anomalousTrafficTest.txt']:
        if not os.path.exists(P('data/raw', f)):
            try:
                urllib.request.urlretrieve(base + f, P('data/raw', f)); print("downloaded", f)
            except Exception as e:
                print("mirror failed for", f, "->", e, "— use Option A or B")
print("data/raw:", os.listdir(P('data/raw')))

### Prepare CSIC 2010
Raw `.txt` files are detected automatically. For a CSV, the cell prints the column names — set the names below and run it again.

> **Important for the paper:** the paper states how CSIC was loaded. The raw files include POST bodies; a CSV loaded with only its URL column does not. The cell records which route you used in `results/data_route.json` — send it with your results so the paper matches.

In [ ]:
CSV_URL_COL    = None     # e.g. 'URL'
CSV_LABEL_COL  = None     # e.g. 'classification'  (values like 0/1 or Normal/Anomalous)
CSV_METHOD_COL = None     # e.g. 'Method'   (None = treat all as GET)
CSV_BODY_COL   = None     # e.g. 'content'  (POST body; None = ignore bodies)

import pandas as pd
csic_parquet = P('data/processed/csic2010.parquet')
raw = os.listdir(P('data/raw'))
if not os.path.exists(csic_parquet):
    if {'normalTrafficTraining.txt', 'anomalousTrafficTest.txt'} <= set(raw):
        sh([f'{R}/prepare_datasets.py', '--source', 'csic_raw',
            '--normal', P('data/raw/normalTrafficTraining.txt'),
            '--anomaly', P('data/raw/anomalousTrafficTest.txt'),
            '--model-input', MODEL_INPUT, '--out', csic_parquet])
        CSIC_ROUTE = 'raw files (POST bodies included)'
    else:
        csvs = [f for f in raw if f.lower().endswith('.csv')]
        assert csvs, "No CSIC files in data/raw — run one option in Section 2."
        cols = list(pd.read_csv(P('data/raw', csvs[0]), nrows=3).columns)
        print("CSV:", csvs[0], "\ncolumns:", cols)
        assert CSV_URL_COL and CSV_LABEL_COL, "Set CSV_URL_COL and CSV_LABEL_COL above from the list printed, then run again."
        args = [f'{R}/prepare_datasets.py', '--source', 'csv', '--input', P('data/raw', csvs[0]),
                '--url-col', CSV_URL_COL, '--label-col', CSV_LABEL_COL, '--dataset', 'csic2010',
                '--model-input', MODEL_INPUT, '--out', csic_parquet]
        if CSV_METHOD_COL: args += ['--method-col', CSV_METHOD_COL]
        if CSV_BODY_COL:   args += ['--body-col', CSV_BODY_COL]
        sh(args)
        CSIC_ROUTE = f'CSV {csvs[0]} (method column: {CSV_METHOD_COL}, body column: {CSV_BODY_COL})'
    json.dump({'csic_route': CSIC_ROUTE}, open(P('results/data_route.json'), 'w'), indent=1)
sh([f'{R}/prepare_datasets.py', '--split', csic_parquet, '--out-dir', P('data/splits/csic2010')],
   out=P('data/splits/csic2010/test.parquet'))

d = pd.read_parquet(csic_parquet)
print(f"\nCSIC after dedup: total={len(d)} benign={(d.label==0).sum()} malicious={(d.label==1).sum()} "
      f"({100*d.label.mean():.1f}% malicious)")
print("Paper currently says 9,492 total, 68.0% malicious — if yours differs, the paper will be updated to YOUR numbers.")

## 3. Optional: transfer corpora

- **SR-BH 2020** (benign + attacks): Harvard Dataverse, "SR-BH 2020 multi-label dataset" (CC0). Download the CSV and upload it here.
- **2025 ModSecurity capture** (attacks only): the Zenodo record linked from *A Thirty-Day Dataset of Malicious HTTP Requests Blocked by OWASP ModSecurity on a Production Web Server* (doi:10.5281/zenodo.17178461).

Upload the CSVs, set the file names, run once to see the columns, set the column names, run again.

In [ ]:
UPLOAD_EXTRA = False
if UPLOAD_EXTRA:
    from google.colab import files
    for name, data in files.upload().items():
        open(P('data/raw', name), 'wb').write(data); print("saved", name)

SRBH_FILE, SRBH_URL_COL, SRBH_LABEL_COL, SRBH_METHOD_COL, SRBH_BODY_COL = None, None, None, None, None
MODSEC_FILE, MODSEC_URL_COL, MODSEC_METHOD_COL, MODSEC_BODY_COL = None, None, None, None

for f in [SRBH_FILE, MODSEC_FILE]:
    if f: print(f, "columns:", list(pd.read_csv(P('data/raw', f), nrows=3).columns))

def prep_csv(name, file, url, label, method, body, attack_only=False):
    out = P(f'data/processed/{name}.parquet')
    args = [f'{R}/prepare_datasets.py', '--source', 'csv', '--input', P('data/raw', file),
            '--url-col', url, '--dataset', name, '--model-input', MODEL_INPUT, '--out', out]
    args += ['--all-malicious'] if attack_only else ['--label-col', label]
    if method: args += ['--method-col', method]
    if body:   args += ['--body-col', body]
    sh(args, out=out)
    sh([f'{R}/prepare_datasets.py', '--split', out, '--out-dir', P(f'data/splits/{name}')],
       out=P(f'data/splits/{name}/test.parquet'))

HAVE_SRBH = bool(SRBH_FILE and SRBH_URL_COL and SRBH_LABEL_COL)
HAVE_MODSEC = bool(MODSEC_FILE and MODSEC_URL_COL)
if HAVE_SRBH:   prep_csv('srbh2020', SRBH_FILE, SRBH_URL_COL, SRBH_LABEL_COL, SRBH_METHOD_COL, SRBH_BODY_COL)
if HAVE_MODSEC: prep_csv('modsec2025', MODSEC_FILE, MODSEC_URL_COL, None, MODSEC_METHOD_COL, MODSEC_BODY_COL, attack_only=True)
print("SR-BH ready:", HAVE_SRBH, "| 2025 corpus ready:", HAVE_MODSEC)

## 4. Train — one model per seed (~15–20 min each on a T4)

In [ ]:
S = P('data/splits/csic2010')
def model_dir(seed, base=BASE_MODEL, data='csic'):
    tag = {'bert-base-uncased': 'bert', 'ehsanaghaei/SecureBERT': 'securebert',
           'distilbert-base-uncased': 'distilbert'}.get(base, base.replace('/', '_'))
    return P(f'models/{tag}_{data}_s{seed}')

for seed in SEEDS:
    md_ = model_dir(seed)
    sh([f'{R}/train.py', '--train', f'{S}/train.parquet', '--val', f'{S}/val.parquet',
        '--model', BASE_MODEL, '--seed', seed, '--epochs', EPOCHS, '--batch', BATCH,
        '--text-col', 'text', '--out', md_], out=f'{md_}/run_meta.json')

## 5. Evaluate — model only, rules only, hybrid (every seed)

In [ ]:
os.makedirs(P('results/csic'), exist_ok=True)
o = P('results/csic')
for seed in SEEDS:
    md_ = model_dir(seed)
    sh([f'{R}/evaluate.py', '--mode', 'model', '--model', md_, '--test', f'{S}/test.parquet',
        '--out', f'{o}/model_s{seed}.json', '--dump-preds', f'{o}/model_s{seed}.csv',
        '--dump-logits', f'{o}/test_logits_s{seed}.npy'], out=f'{o}/model_s{seed}.json')
    sh([f'{R}/evaluate.py', '--mode', 'hybrid', '--logits-in', f'{o}/test_logits_s{seed}.npy', '--model', md_,
        '--test', f'{S}/test.parquet', '--out', f'{o}/hybrid_s{seed}.json',
        '--dump-preds', f'{o}/hybrid_s{seed}.csv'], out=f'{o}/hybrid_s{seed}.json')
    # validation logits, used by the calibration step
    sh([f'{R}/evaluate.py', '--mode', 'model', '--model', md_, '--test', f'{S}/val.parquet',
        '--out', f'{o}/val_s{seed}.json', '--dump-logits', f'{o}/val_logits_s{seed}.npy'], out=f'{o}/val_s{seed}.json')
sh([f'{R}/evaluate.py', '--mode', 'rules', '--test', f'{S}/test.parquet',
    '--out', f'{o}/rules.json', '--dump-preds', f'{o}/rules.csv'], out=f'{o}/rules.json')

for f in sorted(glob.glob(f'{o}/model_s*.json') + glob.glob(f'{o}/hybrid_s*.json')) + [f'{o}/rules.json']:
    show(f)

## 6. Classical baselines (TF-IDF + SVM / RF / LogReg)

In [ ]:
if RUN_BASELINES:
    sh([f'{R}/baselines.py', '--train', f'{S}/train.parquet', '--test', f'{S}/test.parquet',
        '--text-col', 'text', '--out', P('results/csic/baselines.json')], out=P('results/csic/baselines.json'))
    print(json.dumps(json.load(open(P('results/csic/baselines.json'))), indent=1))

## 7. Statistics — mean ± std over seeds, McNemar tests, bootstrap CIs

In [ ]:
if RUN_STATS:
    st, o = f'{R}/stats_tests.py', P('results/csic')
    for mode in ['model', 'hybrid']:
        sh([st, 'aggregate', '--inputs', f'{o}/{mode}_s*.json', '--out', f'{o}/{mode}_agg.json'], force=True)
    s0 = SEEDS[0]
    sh([st, 'mcnemar', '--a', f'{o}/model_s{s0}.csv', '--b', f'{o}/hybrid_s{s0}.csv',
        '--out', f'{o}/mcnemar_model_vs_hybrid.json'], force=True)
    sh([st, 'mcnemar', '--a', f'{o}/rules.csv', '--b', f'{o}/hybrid_s{s0}.csv',
        '--out', f'{o}/mcnemar_rules_vs_hybrid.json'], force=True)
    for mode in ['model', 'hybrid']:
        sh([st, 'bootstrap', '--preds', f'{o}/{mode}_s{s0}.csv', '--out', f'{o}/ci_{mode}_s{s0}.json'], force=True)
    for mode in ['model', 'hybrid']:
        a = json.load(open(f'{o}/{mode}_agg.json'))['metrics']
        print(f"{mode:7s}", {k: v['latex'] for k, v in a.items()})

## 8. Threshold sweep and operating points

In [ ]:
if RUN_THRESHOLD:
    s0 = SEEDS[0]
    out = P(f'results/csic/threshold_s{s0}.csv')
    sh([f'{R}/threshold_sweep.py', '--model', model_dir(s0), '--test', f'{S}/test.parquet',
        '--text-col', 'text', '--out', out], out=out)
    print(json.dumps(json.load(open(P(f'results/csic/threshold_s{s0}_points.json')))['operating_points'], indent=1))

## 9. Cross-dataset transfer (zero-shot) and reverse direction

In [ ]:
if RUN_TRANSFER and (HAVE_SRBH or HAVE_MODSEC):
    o = P('results/transfer'); os.makedirs(o, exist_ok=True)
    for seed in SEEDS:
        for name, ok in [('srbh2020', HAVE_SRBH), ('modsec2025', HAVE_MODSEC)]:
            if not ok:
                continue
            t = P(f'data/splits/{name}/test.parquet')
            sh([f'{R}/evaluate.py', '--mode', 'model', '--model', model_dir(seed), '--test', t,
                '--out', f'{o}/csic2{name}_model_s{seed}.json', '--dump-preds', f'{o}/csic2{name}_model_s{seed}.csv',
                '--dump-logits', f'{o}/csic2{name}_logits_s{seed}.npy'], out=f'{o}/csic2{name}_model_s{seed}.json')
            sh([f'{R}/evaluate.py', '--mode', 'hybrid', '--logits-in', f'{o}/csic2{name}_logits_s{seed}.npy',
                '--model', model_dir(seed), '--test', t, '--out', f'{o}/csic2{name}_hybrid_s{seed}.json',
                '--dump-preds', f'{o}/csic2{name}_hybrid_s{seed}.csv'], out=f'{o}/csic2{name}_hybrid_s{seed}.json')
    for f in sorted(glob.glob(f'{o}/csic2*_s*.json')):
        show(f)
    if HAVE_MODSEC:   # which fusion branch fired on the 2025 corpus
        print("\n2025 corpus, hybrid branch counts:",
              json.load(open(f'{o}/csic2modsec2025_hybrid_s{SEEDS[0]}.json'))['branch_counts'])
else:
    print("Transfer skipped (no extra corpora prepared in Section 3).")

In [ ]:
if RUN_REVERSE and HAVE_SRBH:
    SB = P('data/splits/srbh2020')
    for seed in SEEDS[:3]:     # 3 seeds is enough for the reverse direction
        md_ = model_dir(seed, data='srbh')
        sh([f'{R}/train.py', '--train', f'{SB}/train.parquet', '--val', f'{SB}/val.parquet',
            '--model', BASE_MODEL, '--seed', seed, '--epochs', EPOCHS, '--batch', BATCH, '--out', md_],
           out=f'{md_}/run_meta.json')
        for mode in ['model', 'hybrid']:
            out = P(f'results/transfer/srbh2csic_{mode}_s{seed}.json')
            sh([f'{R}/evaluate.py', '--mode', mode, '--model', md_, '--test', f'{S}/test.parquet', '--out', out], out=out)
            show(out)
else:
    print("Reverse direction skipped (needs SR-BH).")

## 10. Calibration — tests the paper's “calibration, not ranking” explanation

In [ ]:
if RUN_CALIBRATION:
    s0, o = SEEDS[0], P('results/csic')
    evals = ['--eval', f'{o}/test_logits_s{s0}.npy:{S}/test.parquet:csic']
    lg = P(f'results/transfer/csic2srbh2020_logits_s{s0}.npy')
    if HAVE_SRBH and os.path.exists(lg):
        evals += ['--eval', f"{lg}:{P('data/splits/srbh2020/test.parquet')}:srbh"]
    sh([f'{R}/calibration.py', '--fit-logits', f'{o}/val_logits_s{s0}.npy', '--fit-split', f'{S}/val.parquet',
        *evals, '--out', P(f'results/calibration_s{s0}.json'), '--bins-dir', P('results/figs')], force=True)
    T = json.load(open(P(f'results/calibration_s{s0}.json')))['temperature']
    if HAVE_SRBH and os.path.exists(lg):
        t = P('data/splits/srbh2020/test.parquet')
        for mode in ['model', 'hybrid']:
            out = P(f'results/transfer/csic2srbh2020_{mode}_calibrated_s{s0}.json')
            sh([f'{R}/evaluate.py', '--mode', mode, '--logits-in', lg, '--model', model_dir(s0),
                '--test', t, '--temperature', T, '--out', out], force=True)
            show(out)

## 11. Learned fusion vs the fixed rule

In [ ]:
if RUN_LEARNED:
    s0, o = SEEDS[0], P('results/csic')
    sh([f'{R}/evaluate.py', '--mode', 'model', '--model', model_dir(s0), '--test', f'{S}/train.parquet',
        '--out', f'{o}/model_train_s{s0}.json', '--dump-preds', f'{o}/model_train_s{s0}.csv'],
       out=f'{o}/model_train_s{s0}.json')
    for learner in ['logreg', 'tree']:
        sh([f'{R}/learned_fusion.py', '--learner', learner,
            '--train-split', f'{S}/train.parquet', '--train-scores', f'{o}/model_train_s{s0}.csv',
            '--test-split', f'{S}/test.parquet', '--test-scores', f'{o}/model_s{s0}.csv',
            '--out', f'{o}/learned_fusion_{learner}_s{s0}.json'], force=True)

## 12. Rule refresh (optional)
Add updated keyword/pattern lists to `configs/rules_refreshed.json` on your Drive (format: `research/waf_rules.py`, function `load_rules_config`), then run this cell. Leave the lists empty to skip. Base new rules on attack styles you can justify, and describe them in the paper.

In [ ]:
RULES_CFG = P('configs/rules_refreshed.json')
if not os.path.exists(RULES_CFG):
    json.dump({"mode": "extend", "keywords": [], "direct_signatures": [],
               "injection_patterns": [], "encoding_patterns": []}, open(RULES_CFG, 'w'), indent=2)
cfg = json.load(open(RULES_CFG))
if HAVE_MODSEC and any(cfg.get(k) for k in ['keywords', 'direct_signatures', 'injection_patterns', 'encoding_patterns']):
    s0 = SEEDS[0]; t = P('data/splits/modsec2025/test.parquet')
    out = P(f'results/transfer/csic2modsec2025_hybrid_refreshed_s{s0}.json')
    sh([f'{R}/evaluate.py', '--mode', 'hybrid', '--model', model_dir(s0), '--test', t,
        '--rules-config', RULES_CFG, '--out', out], force=True)
    show(out)
else:
    print("Rule refresh skipped (no new rules in configs/rules_refreshed.json, or no 2025 corpus).")

## 13. Adversarial evasion — six fixed mutations, all blocked payloads
For the stronger search-based attack, run the published **WAF-A-MoLE** tool against the live `/analyze` endpoint separately.

In [ ]:
if RUN_ADVERSARIAL:
    s0 = SEEDS[0]
    out = P(f'results/csic/adversarial_hybrid_s{s0}.json')
    sh([f'{R}/adversarial_eval.py', '--model', model_dir(s0), '--test', f'{S}/test.parquet',
        '--sample', 0, '--model-input', MODEL_INPUT, '--out', out], out=out)
    print(json.dumps(json.load(open(out)), indent=1))

## 14. Encoder comparison — SecureBERT and DistilBERT (one seed each)

In [ ]:
if RUN_ENCODERS:
    s0 = SEEDS[0]
    for base in ['ehsanaghaei/SecureBERT', 'distilbert-base-uncased']:
        md_ = model_dir(s0, base=base)
        sh([f'{R}/train.py', '--train', f'{S}/train.parquet', '--val', f'{S}/val.parquet', '--model', base,
            '--seed', s0, '--epochs', EPOCHS, '--batch', BATCH, '--out', md_], out=f'{md_}/run_meta.json')
        tag = os.path.basename(md_)
        for mode in ['model', 'hybrid']:
            out = P(f'results/csic/enc_{tag}_{mode}.json')
            sh([f'{R}/evaluate.py', '--mode', mode, '--model', md_, '--test', f'{S}/test.parquet', '--out', out], out=out)
            show(out)
        lat = P(f'results/latency_offline_{tag}.json')
        sh([f'{R}/latency_benchmark.py', '--mode', 'offline', '--model', md_, '--test', f'{S}/test.parquet',
            '--n', 2000, '--out', lat], out=lat)

## 15. Latency — model inference on this GPU
End-to-end latency through Nginx is measured on your laptop with Docker (see `research/README.md`, Step 8).

In [ ]:
if RUN_LATENCY:
    s0 = SEEDS[0]
    out = P(f'results/latency_offline_bert_s{s0}.json')
    sh([f'{R}/latency_benchmark.py', '--mode', 'offline', '--model', model_dir(s0),
        '--test', f'{S}/test.parquet', '--n', 2000, '--out', out], out=out)
    print(json.dumps(json.load(open(out))['latency'], indent=1))

## 16. Figure data + download everything

In [ ]:
o = P('results/csic')
sh([f'{R}/make_figures.py', '--results', o, '--fig-dir', P('results/figs'),
    '--scores', f'{o}/model_s{SEEDS[0]}.csv:bert', f'{o}/hybrid_s{SEEDS[0]}.csv:hybrid'], force=True)

# zip results + training metadata (not the large model weights)
zip_path = '/content/results_q1.zip'
subprocess.run(['bash', '-c', f'cd "{WORK}" && rm -f {zip_path} && zip -qr {zip_path} results configs models/*/run_meta.json'], check=True)
print("zip size:", round(os.path.getsize(zip_path) / 1e6, 1), "MB")
from google.colab import files
files.download(zip_path)
print("Send results_q1.zip to Claude so the paper can be updated with these numbers.")